# 1) Setup 
(Python 3.11 recommended for torch build)

In [1]:
import numpy as np, pandas as pd, yfinance as yf, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
import timesfm

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.13 (main, Jun  5 2025, 08:21:08) [Clang 14.0.6 ].


/Users/nicolas/Desktop/Repos/zhaw_arep/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2) Load a public checkpoint 
(TimesFM v1.0, 200M, PyTorch)

In [2]:
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend="cpu",            # "gpu" if you have CUDA
        horizon_len=20,           # predict next 20 days
        context_len=256,          # optimal for returns (as shown in debugging)
        use_positional_embedding=False
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-1.0-200m-pytorch"
    ),
)

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 46091.25it/s]


# 3) Get a daily series (SPY close) and create returns


In [3]:
# Download SPY data with auto_adjust=False to get the traditional column structure
spy_data = yf.download("SPY", period="5y", interval="1d", auto_adjust=False)
spy = spy_data["Adj Close"].dropna()
# Use returns - TimesFM works much better with returns (as shown in debugging)
rets = spy.pct_change().dropna()  # simple returns

[*********************100%***********************]  1 of 1 completed


# 4) Build a rolling test
feed last 256 returns, forecast next 20 returns

In [4]:
context = 256
h = 20
y = rets.values.flatten().astype(np.float32)  # use returns
preds, targets = [], []
i = context
while i + h <= len(y):
    window = y[i - context:i]  # now this will be 1D
    # freq code: 0 = high (daily or higher), 1 = weekly/monthly, 2 = quarterly/yearly
    pf, _ = tfm.forecast([window], freq=[0])  # point forecast (and experimental quantiles)
    preds.append(pf[0])
    targets.append(y[i:i+h])
    i += h  # non-overlapping; use i += 1 for a stricter walk-forward

preds = np.concatenate(preds)
targets = np.concatenate(targets)

SyntaxError: invalid syntax (3407635328.py, line 2)

In [ ]:
# DEBUG: Let's investigate what's going wrong
print("=== DATA DEBUGGING ===")
print(f"Original SPY data shape: {spy_data.shape}")
print(f"SPY price range: {float(spy.min()):.2f} to {float(spy.max()):.2f}")
print(f"Price levels shape: {price_levels.shape}")
print(f"Price levels range: {price_levels.min():.2f} to {price_levels.max():.2f}")
print(f"First 10 price levels: {price_levels[:10]}")
print(f"Last 10 price levels: {price_levels[-10:]}")

print(f"\n=== FORECASTING DEBUGGING ===")
print(f"Context length: {context}")
print(f"Horizon length: {h}")
print(f"Total data points: {len(y)}")
print(f"Number of forecast windows: {len(targets) // h}")
print(f"Targets shape: {targets.shape}")
print(f"Predictions shape: {preds.shape}")

print(f"\n=== SAMPLE FORECAST ===")
# Look at the first forecast window
first_window = y[0:context]
first_pred = preds[0:h]
first_target = targets[0:h]
print(f"First context window (last 5): {first_window[-5:]}")
print(f"First prediction: {first_pred}")
print(f"First target: {first_target}")
print(f"Context -> Prediction jump: {first_window[-1]:.2f} -> {first_pred[0]:.2f}")
print(f"Context -> Target jump: {first_window[-1]:.2f} -> {first_target[0]:.2f}")


In [ ]:
# SIMPLE TEST: Let's try TimesFM on a simple synthetic time series
print("=== SIMPLE TIMESFM TEST ===")
# Create a simple upward trending series
simple_series = np.array([100 + i*0.1 for i in range(600)], dtype=np.float32)
print(f"Simple series range: {simple_series.min():.2f} to {simple_series.max():.2f}")

# Test forecast
test_context = simple_series[-512:]  # Last 512 points
test_pred, _ = tfm.forecast([test_context], freq=[0])
print(f"Test prediction: {test_pred[0]}")
print(f"Last context value: {test_context[-1]:.2f}")
print(f"First prediction: {test_pred[0][0]:.2f}")
print(f"Prediction range: {test_pred[0].min():.2f} to {test_pred[0].max():.2f}")

# Check if predictions are reasonable
if test_pred[0][0] < 0:
    print("⚠️  WARNING: TimesFM is predicting negative values!")
if abs(test_pred[0][0] - test_context[-1]) > 50:
    print("⚠️  WARNING: TimesFM prediction is very different from last context value!")


In [ ]:
# CORRECTED IMPLEMENTATION: Let's try with proper data preprocessing
print("=== CORRECTED IMPLEMENTATION ===")

# Option 1: Use returns (as originally intended)
spy_returns = spy.pct_change().dropna().values.flatten().astype(np.float32)
print(f"Returns range: {spy_returns.min():.4f} to {spy_returns.max():.4f}")
print(f"Returns mean: {spy_returns.mean():.4f}, std: {spy_returns.std():.4f}")

# Option 2: Normalize price levels
spy_normalized = (spy - spy.mean()) / spy.std()
spy_normalized = spy_normalized.values.flatten().astype(np.float32)
print(f"Normalized prices range: {spy_normalized.min():.4f} to {spy_normalized.max():.4f}")

# Test both approaches
print("\n--- Testing with returns ---")
context_ret = 256  # Smaller context for returns
h_ret = 20
y_ret = spy_returns
preds_ret, targets_ret = [], []
i = context_ret
while i + h_ret <= len(y_ret):
    window = y_ret[i - context_ret:i]
    pf, _ = tfm.forecast([window], freq=[0])
    preds_ret.append(pf[0])
    targets_ret.append(y_ret[i:i+h_ret])
    i += h_ret

preds_ret = np.concatenate(preds_ret)
targets_ret = np.concatenate(targets_ret)

print(f"Returns MAE: {mean_absolute_error(targets_ret, preds_ret):.4f}")
print(f"Naive 0 MAE: {mean_absolute_error(targets_ret, np.zeros_like(targets_ret)):.4f}")
print(f"Returns prediction range: {preds_ret.min():.4f} to {preds_ret.max():.4f}")
print(f"Returns target range: {targets_ret.min():.4f} to {targets_ret.max():.4f}")


# 5) Evaluate vs two naive baselines

In [ ]:
# Simple evaluation for returns (as shown to work in debugging)
# Naive-0: predict 0 return (random-walk in levels)
naive0 = np.zeros_like(targets)
# Naive-last: predict last observed return
naive_last = np.concatenate([np.repeat(y[context-1], h)] * (len(targets)//h))

def mAE(y_true, y_pred): return mean_absolute_error(y_true, y_pred)

print("=== RETURNS EVALUATION ===")
print("MAE TimesFM (returns):", mAE(targets, preds))
print("MAE Naive 0 (returns):", mAE(targets, naive0))
print("MAE Naive last (returns):", mAE(targets, naive_last))

In [ ]:
# 6) Visualize the forecasting results
import matplotlib.pyplot as plt

# Set up the plot with academic style
plt.style.use('default')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Actual vs Predicted returns (first 200 points for clarity)
n_show = min(200, len(targets))
x_axis = range(n_show)

ax1.plot(x_axis, targets[:n_show], 'k-', linewidth=1, label='Actual', alpha=0.7)
ax1.plot(x_axis, preds[:n_show], 'r-', linewidth=1, label='TimesFM', alpha=0.7)
ax1.plot(x_axis, naive0[:n_show], 'b--', linewidth=1, label='Naive 0', alpha=0.7)
ax1.plot(x_axis, naive_last[:n_show], 'g--', linewidth=1, label='Naive Last', alpha=0.7)

ax1.set_xlabel('Time Steps')
ax1.set_ylabel('Returns')
ax1.set_title('Forecasting Performance Comparison')
ax1.legend(frameon=False)
ax1.grid(True, alpha=0.3)

# Plot 2: MAE comparison
methods = ['TimesFM', 'Naive 0', 'Naive Last']
mae_values = [mAE(targets, preds), mAE(targets, naive0), mAE(targets, naive_last)]
colors = ['red', 'blue', 'green']

bars = ax2.bar(methods, mae_values, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Mean Absolute Error')
ax2.set_title('Model Performance Comparison')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, mae_values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.0005,
             f'{value:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSummary Statistics:")
print(f"Number of predictions: {len(targets)}")
print(f"Actual returns - Mean: {np.mean(targets):.4f}, Std: {np.std(targets):.4f}")
print(f"TimesFM predictions - Mean: {np.mean(preds):.4f}, Std: {np.std(preds):.4f}")
print(f"Best performing method: {methods[np.argmin(mae_values)]}")
